# Save Your Work

Before starting, save this notebook to your Google Drive:
1. Click **File** → **Save a copy in Drive**
2. The copy will open automatically
3. Work in the Google Drive copy from now on

---

# Mushroom Case Study

Apply the full classification workflow — EDA, cleaning, encoding, model training, and threshold tuning — to a safety-critical dataset: distinguishing edible from poisonous mushrooms based on physical features.

This notebook covers all five lessons in module 10:
1. Introduction & Data Loading
2. Exploratory Data Analysis
3. Data Cleaning
4. Feature Engineering
5. Model Selection

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, recall_score, precision_score, f1_score
)

---

# Part 1: Introduction & Data Loading

The **UCI Mushroom Dataset** describes 8,124 hypothetical mushroom samples across 23 species of gilled mushrooms. Every feature is categorical — encoded as single-letter codes. The business question:

> **Can we reliably distinguish edible from poisonous mushrooms from physical features alone — with high enough recall on the poisonous class to be trusted in a safety-critical application?**

The raw file has no header row, so column names must be provided manually.

## Load the Dataset

In [ ]:
columns = [
    "class", "cap_shape", "cap_surface", "cap_color",
    "bruises", "odor", "gill_attachment", "gill_spacing",
    "gill_size", "gill_color", "stalk_shape", "stalk_root",
    "stalk_surface_above", "stalk_surface_below",
    "stalk_color_above", "stalk_color_below",
    "veil_type", "veil_color", "ring_number", "ring_type",
    "spore_print_color", "population", "habitat"
]

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/mushroom/agaricus-lepiota.data"
df = pd.read_csv(url, header=None, names=columns)

print(f"Shape: {df.shape}")
print(f"\nData types:\n{df.dtypes.value_counts()}")
df.head()

## Initial Impressions

In [ ]:
# Class distribution
print("Class distribution:")
print(df["class"].value_counts())
print(f"\nPoisonous fraction: {(df['class'] == 'p').mean():.1%}")

In [ ]:
# Quick scan for missing values (standard NaN)
print(f"Standard missing values: {df.isnull().sum().sum()}")

# Check for the '?' encoding used in this dataset
print("\nCount of '?' per column:")
print((df == "?").sum()[df.columns[(df == "?").sum() > 0]])

In [ ]:
# Check for constant columns (zero-variance features)
for col in df.columns:
    n_unique = df[col].nunique()
    if n_unique == 1:
        print(f"{col}: only 1 unique value → {df[col].unique()}")

**Initial scan summary:**
- 8,124 rows, 23 columns — every column is `object` (categorical)
- Nearly balanced: 52% edible, 48% poisonous
- `stalk_root` uses `?` to encode 2,480 missing values (30.5% of rows)
- `veil_type` has only 1 unique value across all rows → zero-variance feature

---

# Part 2: Exploratory Data Analysis

Before making any changes, understand the data's distributions and which features most strongly separate edible from poisonous mushrooms.

In [ ]:
# Add a numeric label for easier analysis
df["label"] = (df["class"] == "p").astype(int)   # 1 = poisonous, 0 = edible

## Step 1: Feature Cardinality Overview

In [ ]:
cardinality = df.drop(columns=["class", "label"]).nunique().sort_values()
print(cardinality)

## Step 2: Poisonous Rate by Feature Value

For each feature value, what fraction of mushrooms with that value are poisonous? A feature where some values are 0% poisonous and others are 100% is perfectly discriminative.

In [ ]:
def poisonous_rate(df, column):
    """Compute poisonous fraction for each value of a column."""
    return (
        df.groupby(column)["label"]
        .agg(["mean", "count"])
        .rename(columns={"mean": "poisonous_rate", "count": "n"})
        .sort_values("poisonous_rate", ascending=False)
        .round(3)
    )

## Step 3: The Most Predictive Feature — Odor

In [ ]:
print(poisonous_rate(df, "odor"))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
odor_rate = poisonous_rate(df, "odor").reset_index()
colors = ["#e74c3c" if r > 0.5 else "#2ecc71" for r in odor_rate["poisonous_rate"]]
ax.bar(odor_rate["odor"], odor_rate["poisonous_rate"], color=colors)
ax.set_title("Poisonous Rate by Odor")
ax.set_xlabel("Odor code")
ax.set_ylabel("Fraction poisonous")
ax.axhline(0.5, color="black", linestyle="--", linewidth=0.8)
plt.tight_layout()
plt.show()

## Step 4: Spore Print Color

In [ ]:
print(poisonous_rate(df, "spore_print_color"))

## Step 5: Gill Color

In [ ]:
print(poisonous_rate(df, "gill_color"))

## Step 6: Features with Weaker Signal

In [ ]:
print("Cap color:")
print(poisonous_rate(df, "cap_color"))

In [ ]:
print("Habitat:")
print(poisonous_rate(df, "habitat"))

## Step 7: The stalk_root Column

In [ ]:
print(poisonous_rate(df, "stalk_root"))

The `?` category (missing values) has a poisonous rate of ~57% — higher than the dataset average of 48.2%. The missingness itself is informative. Do **not** simply drop rows with `?`.

## Step 8: Visualizing Class Separation

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
features = ["odor", "spore_print_color", "gill_color", "stalk_root", "habitat", "ring_type"]

for ax, feat in zip(axes.flat, features):
    ct = pd.crosstab(df[feat], df["class"], normalize="index")
    ct.plot(kind="bar", stacked=True, ax=ax,
            color={"e": "#2ecc71", "p": "#e74c3c"}, width=0.8)
    ax.set_title(f"{feat}")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=45)
    ax.legend(["Edible", "Poisonous"], fontsize=8, loc="upper right")

plt.suptitle("Edible vs. Poisonous by Feature Value", y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

## Step 9: Cramér's V — Feature Association with Target

For categorical features, Cramér's V measures association (analogous to correlation, ranging from 0 to 1).

In [ ]:
def cramers_v(x, y):
    confusion = pd.crosstab(x, y)
    chi2 = chi2_contingency(confusion)[0]
    n = confusion.sum().sum()
    phi2 = chi2 / n
    r, k = confusion.shape
    return np.sqrt(phi2 / min(r - 1, k - 1))

features = [c for c in df.columns if c not in ("class", "label")]
associations = {f: cramers_v(df[f], df["class"]) for f in features}
assoc_series = pd.Series(associations).sort_values(ascending=False)
print(assoc_series.round(3))

**EDA Summary:**

| Finding | Implication |
|---------|-------------|
| `odor` is nearly perfectly discriminative (Cramér's V ≈ 0.977) | A shallow decision tree on odor alone will likely exceed 95% accuracy |
| `spore_print_color`, `gill_color`, `ring_type` are next strongest | These drive boundaries beyond odor |
| `stalk_root` has 2,480 `?` values; missingness correlates with class | Keep as feature, treat `?` as its own category |
| `veil_type` has exactly one unique value | Zero information; drop unconditionally |
| All features are categorical | Every feature must be encoded before modeling |

---

# Part 3: Data Cleaning

Two tasks identified by EDA: drop `veil_type` (zero variance) and handle `?` in `stalk_root` (30% of rows).

In [ ]:
# Always work on a copy to preserve the raw data
df_clean = df.drop(columns=["label"]).copy()
print(f"Starting shape: {df_clean.shape}")

## Step 1: Confirm No Standard Missing Values

In [ ]:
print(f"Standard NaN count: {df_clean.isnull().sum().sum()}")

## Step 2: Check for Duplicate Rows

In [ ]:
n_dupes = df_clean.duplicated().sum()
print(f"Duplicate rows: {n_dupes}")

## Step 3: Drop veil_type

In [ ]:
print(f"veil_type unique values: {df_clean['veil_type'].unique()}")
print(f"veil_type value counts:\n{df_clean['veil_type'].value_counts()}")

In [ ]:
df_clean = df_clean.drop(columns=["veil_type"])
print(f"Shape after dropping veil_type: {df_clean.shape}")

## Step 4: Handle stalk_root — The ? Values

Three strategies:

| Strategy | Action | Tradeoff |
|----------|--------|----------|
| **Drop rows** | Remove all 2,480 rows with `?` | Lose 30% of data; potential bias |
| **Impute with mode** | Replace `?` with `b` (most common) | Artificially inflates the `b` category |
| **Keep `?` as its own category** | Treat missing as value `m` | Preserves all data; treats missingness as informative |

The `?` rows have a poisonous rate of 56.8% — the missingness is **not random**. Keeping it as its own category preserves this signal.

In [ ]:
print(f"stalk_root value counts before replacement:")
print(df_clean["stalk_root"].value_counts())
print(f"\nMissing (?): {(df_clean['stalk_root'] == '?').sum()} rows "
      f"({(df_clean['stalk_root'] == '?').mean():.1%})")

In [ ]:
df_clean["stalk_root"] = df_clean["stalk_root"].replace("?", "m")   # m = missing

print(f"stalk_root after replacement:")
print(df_clean["stalk_root"].value_counts())

## Step 5: Verify All Remaining Columns

In [ ]:
print("Unique values per column:")
for col in df_clean.drop(columns=["class"]).columns:
    vals = sorted(df_clean[col].unique())
    print(f"  {col:25s}: {vals}")

## Step 6: Confirm Target Column

In [ ]:
print(f"Target column (class) values: {df_clean['class'].unique()}")
print(f"Class distribution:")
print(df_clean["class"].value_counts())

## Step 7: Re-check for Zero-Variance Columns

In [ ]:
for col in df_clean.columns:
    if df_clean[col].nunique() < 2:
        print(f"WARNING: {col} has {df_clean[col].nunique()} unique value(s)")

print("Zero-variance check complete.")

In [ ]:
print(f"Final shape: {df_clean.shape}")
print(f"Missing values (NaN): {df_clean.isnull().sum().sum()}")
print(f"Missing values (?): {(df_clean == '?').sum().sum()}")

**Cleaning summary:**

| Step | Change | Rationale |
|------|--------|----------|
| Drop `veil_type` | Removed 1 column | Zero variance — carries no information |
| Replace `?` with `m` in `stalk_root` | 2,480 values changed | Missingness is informative; treat as its own category |
| Deduplication | No rows removed | No duplicates present |

---

# Part 4: Feature Engineering

Every feature is still a string. Encode all 21 feature columns into integers, then make the train/test split.

## Step 1: Encode the Target

In [ ]:
y = (df_clean["class"] == "p").astype(int)   # 1 = poisonous, 0 = edible
X = df_clean.drop(columns=["class"])

print(f"Target distribution:")
print(y.value_counts())
print(f"\nFeature matrix shape: {X.shape}")
print(f"Feature columns ({len(X.columns)}): {list(X.columns)}")

## Step 2: Train/Test Split First

Always split before encoding — fit transformers on training data only, then apply to test data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"Poisonous rate — train: {y_train.mean():.1%}, test: {y_test.mean():.1%}")

## Step 3: Apply Label Encoding

Fit each encoder on training data only, then transform both train and test.

In [ ]:
X_train_enc = X_train.copy()
X_test_enc  = X_test.copy()

encoders = {}

for col in X_train.columns:
    le = LabelEncoder()
    X_train_enc[col] = le.fit_transform(X_train[col])
    X_test_enc[col]  = le.transform(X_test[col])
    encoders[col]    = le   # save for later use

print("Encoding complete.")
print(f"\nX_train_enc dtypes:\n{X_train_enc.dtypes.value_counts()}")
print(f"\nFirst 3 rows (encoded):")
print(X_train_enc.head(3))

## Step 4: Check for Unseen Categories in the Test Set

In [ ]:
issues = []
for col in X_train.columns:
    train_vals = set(X_train[col].unique())
    test_vals  = set(X_test[col].unique())
    unseen = test_vals - train_vals
    if unseen:
        issues.append((col, unseen))

if issues:
    for col, vals in issues:
        print(f"WARNING: {col} has unseen test values: {vals}")
else:
    print("No unseen categories in test set.")

## Step 5: Feature Summary

In [ ]:
summary = pd.DataFrame({
    "column": X_train.columns,
    "n_unique_values": [X_train[c].nunique() for c in X_train.columns],
    "encoded_range": [f"0–{X_train_enc[c].max()}" for c in X_train.columns],
}).set_index("column")

print(summary)

## Step 6: Sanity Checks

In [ ]:
assert X_train_enc.isnull().sum().sum() == 0, "Unexpected NaN in training features"
assert X_test_enc.isnull().sum().sum() == 0, "Unexpected NaN in test features"
assert X_train_enc.shape == (6499, 21)
assert X_test_enc.shape  == (1625, 21)

print("All checks passed.")
print(f"\nFinal feature matrix shapes:")
print(f"  X_train_enc: {X_train_enc.shape}")
print(f"  X_test_enc:  {X_test_enc.shape}")

---

# Part 5: Model Selection

**Evaluation priority (in this order):**
1. **Recall on poisonous class** — missing a poisonous mushroom is the worst error
2. **F1-score on poisonous class** — balances precision against recall
3. **AUC** — overall separability, threshold-independent
4. **Accuracy** — useful summary but not the primary lens

## Model 1: Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_enc, y_train)

y_pred_lr   = lr.predict(X_test_enc)
y_proba_lr  = lr.predict_proba(X_test_enc)[:, 1]

print("=== Logistic Regression ===")
print(classification_report(y_test, y_pred_lr, target_names=["Edible", "Poisonous"]))
print(f"AUC: {roc_auc_score(y_test, y_proba_lr):.4f}")

## Model 2: Decision Tree

In [ ]:
dt = DecisionTreeClassifier(max_depth=6, random_state=42)
dt.fit(X_train_enc, y_train)

y_pred_dt   = dt.predict(X_test_enc)
y_proba_dt  = dt.predict_proba(X_test_enc)[:, 1]

print("=== Decision Tree (max_depth=6) ===")
print(classification_report(y_test, y_pred_dt, target_names=["Edible", "Poisonous"]))
print(f"AUC: {roc_auc_score(y_test, y_proba_dt):.4f}")

In [ ]:
# Feature importances from the decision tree
print("Top 5 most important features (Decision Tree):")
importances = pd.Series(dt.feature_importances_, index=X_train_enc.columns)
print(importances.sort_values(ascending=False).head(5).round(4))

In [ ]:
# Visualize the top of the tree
plt.figure(figsize=(16, 6))
plot_tree(
    dt,
    max_depth=2,
    feature_names=X_train_enc.columns,
    class_names=["Edible", "Poisonous"],
    filled=True,
    fontsize=9
)
plt.title("Decision Tree — Top 2 Levels")
plt.tight_layout()
plt.show()

## Model 3: Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_enc, y_train)

y_pred_rf   = rf.predict(X_test_enc)
y_proba_rf  = rf.predict_proba(X_test_enc)[:, 1]

print("=== Random Forest (100 trees) ===")
print(classification_report(y_test, y_pred_rf, target_names=["Edible", "Poisonous"]))
print(f"AUC: {roc_auc_score(y_test, y_proba_rf):.4f}")

## Model Comparison

In [ ]:
results = []
for name, pred, proba in [
    ("Logistic Regression", y_pred_lr, y_proba_lr),
    ("Decision Tree",       y_pred_dt, y_proba_dt),
    ("Random Forest",       y_pred_rf, y_proba_rf),
]:
    results.append({
        "Model":               name,
        "Accuracy":            f"{(pred == y_test).mean():.3f}",
        "Poisonous Recall":    f"{recall_score(y_test, pred):.3f}",
        "Poisonous Precision": f"{precision_score(y_test, pred):.3f}",
        "Poisonous F1":        f"{f1_score(y_test, pred):.3f}",
        "AUC":                 f"{roc_auc_score(y_test, proba):.4f}",
    })

print(pd.DataFrame(results).to_string(index=False))

## Tuning the Decision Threshold for Logistic Regression

Even when decision trees are perfect on this dataset, threshold tuning on logistic regression shows how the recall/precision trade-off works for real-world problems with less clean separation.

In [ ]:
thresholds = [0.5, 0.4, 0.3, 0.2, 0.1]
print(f"{'Threshold':<12} {'Recall':>8} {'Precision':>10} {'F1':>8} {'FN (missed poisonous)':>22}")
print("-" * 64)

for t in thresholds:
    preds = (y_proba_lr >= t).astype(int)
    rec   = recall_score(y_test, preds)
    prec  = precision_score(y_test, preds)
    f1    = f1_score(y_test, preds)
    fn    = ((preds == 0) & (y_test == 1)).sum()
    print(f"{t:<12.1f} {rec:>8.3f} {prec:>10.3f} {f1:>8.3f} {fn:>22}")

In [ ]:
# Confusion matrix at threshold 0.1
y_pred_lr_low = (y_proba_lr >= 0.1).astype(int)
cm = confusion_matrix(y_test, y_pred_lr_low)
disp = ConfusionMatrixDisplay(cm, display_labels=["Edible", "Poisonous"])
disp.plot(cmap="Greens")
plt.title("Logistic Regression @ threshold=0.1")
plt.show()

## Feature Importance Comparison

In [ ]:
dt_imp = pd.Series(dt.feature_importances_, index=X_train_enc.columns)
rf_imp = pd.Series(rf.feature_importances_, index=X_train_enc.columns)
lr_imp = pd.Series(np.abs(lr.coef_[0]), index=X_train_enc.columns)
lr_imp = lr_imp / lr_imp.sum()

top_n = 8
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, imp) in zip(axes, [
    ("Decision Tree", dt_imp),
    ("Random Forest", rf_imp),
    ("Logistic Regression (|coef|)", lr_imp),
]):
    top = imp.sort_values(ascending=False).head(top_n)
    ax.barh(top.index[::-1], top.values[::-1], color="steelblue")
    ax.set_title(name)
    ax.set_xlabel("Importance")

plt.suptitle("Top Feature Importances by Model", fontsize=13)
plt.tight_layout()
plt.show()

## Cross-Validation on the Final Model

In [ ]:
cv_scores = cross_val_score(
    DecisionTreeClassifier(max_depth=6, random_state=42),
    X_train_enc, y_train,
    cv=5, scoring="recall"
)
print(f"5-fold CV recall (poisonous): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

## Final Model Recommendation

| Model | Accuracy | Poisonous Recall | Interpretability | Deployment |
|-------|----------|-----------------|-----------------|------------|
| Logistic Regression (t=0.1) | ~95% | 100% | Medium | Simple |
| Decision Tree (depth=6) | 100% | 100% | High — rules readable | Simple |
| Random Forest | 100% | 100% | Low — many trees | Moderate |

**Recommendation: Decision Tree with `max_depth=6`**
- Achieves perfect recall without threshold tuning
- Decision rules are fully interpretable
- Simpler and faster than a random forest
- 5-fold cross-validation confirms the result is not a lucky split